<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-header">

## Tutorial \#8: Sharding

</div>

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Although <b><i>Spark</i></b> is relatively fast, some pipelines simply require more power in order to achieve acceptable performances or even to be able to run them to begin with. This is often achieved by distributing the computation across multiple devices. 

Often, transforming a model from a single-device paradigm into a multi-device paradigm can be challenging and extremely time consuming. However this can be easily achieved in <b><i>Spark</i></b> thanks to JAX's sharding system. In this tutorial we will not go deeper into the details of sharding since JAX already does a fantastic job, but for a detailed explanation on sharding we refer the reader to: 

1. [Distributed arrays and automatic parallelization](https://docs.jax.dev/en/latest/notebooks/Distributed_arrays_and_automatic_parallelization.html)
2. [Explicit sharding](https://docs.jax.dev/en/latest/notebooks/explicit-sharding.html)


Let's start by instructing XLA to simulate 6 physical devices!

</div>

In [1]:
# Set XLA flags
import os
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=6'

# Indicate Jax to use CPU's
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

import jax
import jax.numpy as jnp
print(jax.devices())

[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5)]


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Let's also import the model from the previous tutorial.

</div>

In [2]:
import spark

NEURON_PATH = './alif_mod_neuron.scfg'
BRAIN_PATH = './ab_brain.scfg'

# Load custom neuronal model
try:
	spark.register_neuron_from_config_file('ALIFModNeuron', NEURON_PATH)
except Exception as e:
	print(f'(File) Oh no: {e}!')

In [3]:
# Load the Brain.
brain_config = spark.nn.BrainConfig.from_file(BRAIN_PATH)

# Initialize the Brain.
brain = spark.nn.Brain(config=brain_config)

# Build the Brain.
brain(
    signal=spark.FloatArray(jnp.zeros((4,), dtype=jnp.float16)),
    mod_a_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_a_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
)


{'action': FloatArray(value=Array([0., 0.], dtype=float16))}

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

By default, JAX arrays and, as a consequence, <b><i>Spark</i></b> models are not commited to any particular device. To see this, we can inspect the internal values of the brain to see where the variables are located. Since there are many many variables inside a Brain, let's just look at three of them.

</div>

In [4]:
print('Array: brain.A_excitatory.soma.potential')
jax.debug.visualize_array_sharding(brain.A_excitatory.soma.potential)

print('Array: brain.B_excitatory.synapses.kernel')
jax.debug.visualize_array_sharding(brain.B_excitatory.synapses.kernel)

print('Array: brain.integrator.trace.tracer_rise.trace')
jax.debug.visualize_array_sharding(brain.integrator.trace.tracer_rise.trace)

print('Array: brain._cache["B_inhibitory", "out_spikes"]')
jax.debug.visualize_array_sharding(brain._cache['B_inhibitory', 'out_spikes'].value)

Array: brain.A_excitatory.soma.potential


  CPU 0  
         

Array: brain.B_excitatory.synapses.kernel


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                     CPU 0                                      
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                

Array: brain.integrator.trace.tracer_rise.trace


   CPU 0    
            

Array: brain._cache["B_inhibitory", "out_spikes"]


  CPU 0  
         

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

As we can see everything is located in the CPU 0. This happens because we have not let JAX know that we wish to use multiple devices. In JAX we often use a <b>Mesh</b> (of devices) to let the JIT compiler know how we want to distribute the data. This is a simple way to define how to paralellize your pipeline across different abstract axes, e.g., the bacth and the model. 

In our case we don't have batches, so let's just start by distributing the model across all devices. This can be easily achieved by creating a mesh and letting JAX know that we want to use this mesh of devices.

</div>

In [5]:
# Create mesh
auto_mesh = jax.sharding.Mesh(
    jax.devices(), 
    axis_names=('device'), 
    axis_types=(jax.sharding.AxisType.Auto,)
)

# Set mesh
jax.set_mesh(auto_mesh)

# Rebuild the Brain.
brain = spark.nn.Brain(config=brain_config)
brain(
    signal=spark.FloatArray(jnp.zeros((4,), dtype=jnp.float16)),
    mod_a_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_a_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
)

{'action': FloatArray(value=Array([0., 0.], dtype=float16))}

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

We can now inspect the brain again, and see that everything is now in all our devices!

</div>

In [6]:
print('Array: brain.A_excitatory.soma.potential')
jax.debug.visualize_array_sharding(brain.A_excitatory.soma.potential)

print('Array: brain.B_excitatory.synapses.kernel')
jax.debug.visualize_array_sharding(brain.B_excitatory.synapses.kernel)

print('Array: brain.integrator.trace.tracer_rise.trace')
jax.debug.visualize_array_sharding(brain.integrator.trace.tracer_rise.trace)

print('Array: brain._cache["B_inhibitory", "out_spikes"]')
jax.debug.visualize_array_sharding(brain._cache['B_inhibitory', 'out_spikes'].value)

Array: brain.A_excitatory.soma.potential


CPU 0,1,2,3,4,5
               

Array: brain.B_excitatory.synapses.kernel


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                CPU 0,1,2,3,4,5                                 
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                

Array: brain.integrator.trace.tracer_rise.trace


CPU 0,1,2,3,4,5
               

Array: brain._cache["B_inhibitory", "out_spikes"]


CPU 0,1,2,3,4,5
               

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

However, it is important to note that this is not necessarily a good layout. When working with multiple devices, communication matters; a bad distribution of devices may lead to no gain at all from having multiple devices. Therefore, in some cases it may be beneficial to manually assign the devices to sections of our model.

Fortunately this can still be easily achieved, although a little bit more convoluted. For simplicity, let's map all the arrays within a <b><i>Spark</i></b> module to the same devices and everything else to all the devices. 

</div>

In [7]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec

# Shard to device maps.
devices = jax.devices()
shard_to_all_devices  = NamedSharding(Mesh(devices, axis_names=('device')), PartitionSpec())
shard_to_device = [
	NamedSharding(Mesh([devices[i]], axis_names=('device')), PartitionSpec()) for i in range(len(devices))
]

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Next, we need to create a map between arrays and shards, but before that lets explore how this map looks like. We achieve this through the method <b>tree_map_with_path</b>. This allow us to create a map for a pytree using the structure of such pytree. However, in order to use that function, we need to split our model into its graph and its state component.

Now, let's create a dummy map to None just to print all leaf nodes in our pytree, which correspond to all the arrays in our model. The thing to notice here is the path, which we can use to distribute our computation across devices.
</div>

In [8]:
def dummy_map(path: tuple[jax.tree_util.DictKey | jax.tree_util.GetAttrKey], leaf: jax.Array):
	leaf_path = '.'.join([str(p.key) for p in path if hasattr(p, 'key')])
	# Print the path of the leaf
	print(leaf_path)
	return None

graph, state = spark.split((brain))
sharding_rules = jax.tree_util.tree_map_with_path(dummy_map, state)

A_excitatory.delays._bitmask
A_excitatory.delays._current_idx
A_excitatory.delays.rng
A_excitatory.drift_trace.rng
A_excitatory.drift_trace.trace.rng
A_excitatory.drift_trace.trace.trace
A_excitatory.plasticity.post_trace.rng
A_excitatory.plasticity.post_trace.trace
A_excitatory.plasticity.pre_trace.rng
A_excitatory.plasticity.pre_trace.trace
A_excitatory.plasticity.rng
A_excitatory.reward_trace.rng
A_excitatory.reward_trace.trace.rng
A_excitatory.reward_trace.trace.trace
A_excitatory.rng
A_excitatory.soma._potential
A_excitatory.soma.rng
A_excitatory.soma.spike_counter
A_excitatory.soma.threshold_trace.rng
A_excitatory.soma.threshold_trace.trace
A_excitatory.synapses._kernel
A_excitatory.synapses.current_tracer.rng
A_excitatory.synapses.current_tracer.trace
A_excitatory.synapses.rng
A_inhibitory.delays._bitmask
A_inhibitory.delays._current_idx
A_inhibitory.delays.rng
A_inhibitory.drift_trace.rng
A_inhibitory.drift_trace.trace.rng
A_inhibitory.drift_trace.trace.trace
A_inhibitory.plast

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

In our current context it makes sense to allocate specific <b><i>Spark</i></b> modules to specific devices, since most modules take similar times to compute and once the input of a module is given most of the computation is performed locally, so no external variables should be required past the input. 

As we can see from the previous dummy map, we can use the name of the main modules of the brain to create this mapping.

</div>

In [9]:
def get_sharding_map(path: tuple[jax.tree_util.DictKey | jax.tree_util.GetAttrKey], leaf: jax.Array) -> NamedSharding:
	leaf_path = [str(p.key) for p in path if hasattr(p, 'key')]
	# Sharding rules
	if 'spiker' in leaf_path[0]:
		return shard_to_device[0]
	elif 'integrator' in leaf_path[0]:
		return shard_to_device[1]
	elif 'A_excitatory' in leaf_path[0]:
		return shard_to_device[2]
	elif 'A_inhibitory' in leaf_path[0]:
		return shard_to_device[3]
	elif 'B_excitatory' in leaf_path[0]:
		return shard_to_device[4]
	elif 'B_inhibitory' in leaf_path[0]:
		return shard_to_device[5]
	else:
		return shard_to_all_devices


# Get sharding map
graph, state = spark.split((brain))
sharding_map = jax.tree_util.tree_map_with_path(get_sharding_map, state)

# Apply the sharding map
state = jax.device_put(state, sharding_map)

# Reconstruct the brain
brain = spark.merge(graph, state)

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Finally, let's just see the final split of our model!

</div>

In [10]:
print('Array: brain.A_excitatory.soma.potential')
jax.debug.visualize_array_sharding(brain.A_excitatory.soma.potential)

print('Array: brain.B_excitatory.synapses.kernel')
jax.debug.visualize_array_sharding(brain.B_excitatory.synapses.kernel)

print('Array: brain.integrator.trace.tracer_rise.trace')
jax.debug.visualize_array_sharding(brain.integrator.trace.tracer_rise.trace)

print('Array: brain._cache["B_inhibitory", "out_spikes"]')
jax.debug.visualize_array_sharding(brain._cache['B_inhibitory', 'out_spikes'].value)

Array: brain.A_excitatory.soma.potential


  CPU 2  
         

Array: brain.B_excitatory.synapses.kernel


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                     CPU 4                                      
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                

Array: brain.integrator.trace.tracer_rise.trace


   CPU 1    
            

Array: brain._cache["B_inhibitory", "out_spikes"]


CPU 0,1,2,3,4,5
               

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Neat! Every module now lives in its own device. However, there is a catch: placing the arrays is not enough to run the model. JIT compiles one single program for all the devices of its arguments, and it refuses arguments that are committed to different devices. Let's try to run a step and see what happens.

Note that we first need to leave the mesh we set before, since from now on each device is going to run a program of its own.

</div>

In [11]:
from functools import partial
import numpy as np

# Leave the mesh we set before.
jax.set_mesh(None)

@partial(spark.jit, static_argnames=['steps'])
def run(graph, state, steps, **inputs):
    def step(state, _):
        model = spark.merge(graph, state)
        out = model(**inputs)
        _, state = spark.split(model)
        return state, out
    state, outs = spark.scan(step, state, None, length=steps)
    return jax.tree.map(lambda x: x[-1], outs), state

inputs = dict(
    signal=spark.FloatArray(jnp.ones((4,), dtype=jnp.float16)),
    mod_a_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_a_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    mod_b_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_ex=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
    drift_in=spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
)

try:
    run(graph, state, 1, **inputs)
except Exception as error:
    print(f'{type(error).__name__}: {str(error).splitlines()[0]}')

ValueError: Received incompatible devices for jitted computation. Got argument state['A_excitatory']['delays']['_bitmask'].value of run with shape uint8[5,104] and with device ids [2] on platform CPU and argument state['A_inhibitory']['delays']['_bitmask'].value of run with shape uint8[5,96] and with device ids [3] on platform CPU


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

As expected, JAX is not happy with our layout. Running different modules on different devices requires one program per device, and this is precisely what <b>spark.Partition</b> does. A partition divides a brain by its modules and gives each device a sub-brain, that is, a regular <b>spark.nn.Brain</b> that holds only the modules assigned to that device.

Remember from Tutorial \#4 that every module of a brain is updated in parallel, at the cost of one extra timestep? Every module reads what the others produced on the previous step, so the sub-brains do not need to wait for each other within a step: each device runs its own sub-brain and, once all of them are done, the outputs that cross devices are copied to the devices reading them.

Let's use the same assignment as before.

</div>

In [12]:
devices = jax.devices()
partition = spark.Partition(brain, {
    devices[0]: ['spiker'],
    devices[1]: ['integrator'],
    devices[2]: ['A_excitatory'],
    devices[3]: ['A_inhibitory'],
    devices[4]: ['B_excitatory'],
    devices[5]: ['B_inhibitory'],
})

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Running a partition is very similar to running a brain. <b>split</b> gives us the graph and the state of every sub-brain, each state already placed on its device, and <b>initial</b> gives us what each sub-brain reads from the others on its first step. Then, a step of the brain is just a step of every sub-brain, which <b>partition.run</b> takes care of, followed by an <b>exchange</b> of the outputs. Since every call returns as soon as it is dispatched, all devices run at the same time.

</div>

In [13]:
graphs, states = partition.split(brain)
received = partition.initial(states)

for _ in range(100):
    outputs, states = partition.run(run, graphs, states, received, inputs, steps=1)
    received = partition.exchange(outputs)

partition.outputs(outputs)

{'action': FloatArray(value=Array([0.3894, 0.2834], dtype=float16))}

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Let's check that every sub-brain stayed on its own device.

</div>

In [14]:
print('Array: A_excitatory.soma.potential, in the sub-brain of device 2')
jax.debug.visualize_array_sharding(spark.merge(graphs[devices[2]], states[devices[2]]).A_excitatory.soma.potential)

print('Array: B_excitatory.synapses.kernel, in the sub-brain of device 4')
jax.debug.visualize_array_sharding(spark.merge(graphs[devices[4]], states[devices[4]]).B_excitatory.synapses.kernel)

Array: A_excitatory.soma.potential, in the sub-brain of device 2


  CPU 2  
         

Array: B_excitatory.synapses.kernel, in the sub-brain of device 4


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                     CPU 4                                      
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Great, but is it still the same brain? It should be, since the partition does not change the model at all, so let's check it! <b>merge</b> gathers the sub-brains back into a single brain on one device, which we can compare against the very same brain running on a single device for the same 100 steps.

</div>

In [15]:
graph, state = spark.split(brain)
state = jax.device_put(state, devices[0])
for _ in range(100):
    reference, state = run(graph, state, 1, **inputs)

merged = partition.merge(states)
same = lambda a, b: all(np.array_equal(np.asarray(x), np.asarray(y)) for x, y in zip(jax.tree.leaves(a), jax.tree.leaves(b)))
print('Same outputs:', same(reference, partition.outputs(outputs)))
print('Same state:', same(spark.split(merged)[1], state))

Same outputs: True
Same state: True


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Partitioned brains can be recorded just like in Tutorial \#7. While a recorder is open, the calls that <b>partition.run</b> makes on every device count as one single call of the recorder: each device records the probes of its own modules, and the records are joined before they are written. In other words, the run looks exactly like the run of the brain.

</div>

In [16]:
import tempfile
rec = spark.recording

measurements = [
    rec.Measurements('activity', (
        rec.SummaryProbe('A_excitatory.soma:spikes', reduce=('active_fraction',)),
        rec.SummaryProbe('B_excitatory.soma:spikes', reduce=('active_fraction',)),
        rec.SummaryProbe('A_excitatory.soma.potential', reduce=('mean',)),
    ), group=50, trigger=rec.Always()),
]

graphs, states = partition.split(brain)
received = partition.initial(states)
with tempfile.TemporaryDirectory() as root:
    with rec.Recorder(root, brain, measurements, name='sharding') as recorder:
        for _ in range(200):
            outputs, states = partition.run(run, graphs, states, received, inputs, steps=1)
            received = partition.exchange(outputs)
    saved = rec.load(recorder.path)
    for key in saved.scalar_keys():
        steps, values = saved.scalar(key)
        print(key, np.round(values, 4))

activity/A_excitatory.soma.potential/mean [ 5.981  10.9145 10.8708 10.8076]
activity/A_excitatory.soma:spikes/active_fraction [0.0008 0.0016 0.001  0.0018]
activity/B_excitatory.soma:spikes/active_fraction [0.0004 0.001  0.001  0.0006]


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Choosing the parts by hand is not mandatory either. <b>Partition.balanced</b> weighs every module by the size of its state, keeps together the modules that read or write the properties of each other, and places them so that every device carries a similar amount of work while as few outputs as possible cross devices. A device that would only add more communication is simply left out.

</div>

In [17]:
balanced = spark.Partition.balanced(brain, devices)
for device, names in balanced.parts.items():
    print(device, names)

cpu:0 ('A_inhibitory', 'B_inhibitory', 'integrator')
cpu:1 ('A_excitatory', 'spiker')
cpu:2 ('B_excitatory',)


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

So far, we have been working under the hypothesis that all of our devices live in the same machine. However, one of the main reason to partition a brain is to go beyond what a single machine can do.

In a cluster, every machine runs a copy of the same Python program, called a process. Importantly, each process can only see the devices on its own machine so, before computing anything, the processes need to find each other. Fortunately, <b>jax.distributed.initialize</b> does exactly this; this method allows every process to contact a coordinator, which, in a nutshell, allows processes to speak with each other. Overall, this have almost zero setup from our side.

But let's try to illustrate what does this means for an average pipeline. Rather than a cluster, we can illustrate what this means with two processes on the same computer (playing the role of two machines), each with three devices. The easiest way to achieve this is by using an auxiliary file, which we will outside of the notebook. The code is basically the same code we used before, with the call to <b>jax.distributed.initialize</b> at the top.

</div>

In [18]:
%%writefile cluster.py
import os
import sys

# Simulates three devices.
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=3'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

# "Join the cluster" (before computing anything). 
# Every process contacts the coordinator, process 0, at its address.
import jax
process_id = int(sys.argv[1])
jax.distributed.initialize(coordinator_address='localhost:12345', num_processes=2, process_id=process_id)

# Brain definition
from functools import partial
import numpy as np
import jax.numpy as jnp
import spark

spark.register_neuron_from_config_file('ALIFModNeuron', './alif_mod_neuron.scfg')
brain = spark.nn.Brain(config=spark.nn.BrainConfig.from_file('./ab_brain.scfg'))
inputs = {
	'signal':spark.FloatArray(jnp.ones((4,), dtype=jnp.float16)),
	'mod_a_ex':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
	'mod_a_in':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
	'mod_b_ex':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
	'mod_b_in':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
	'drift_ex':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
	'drift_in':spark.FloatArray(jnp.array([0], dtype=jnp.float16)),
}
brain(**inputs)

@partial(spark.jit, static_argnames=['steps'])
def run(graph, state, steps, **inputs):
    def step(state, _):
        model = spark.merge(graph, state)
        out = model(**inputs)
        _, state = spark.split(model)
        return state, out
    state, outs = spark.scan(step, state, None, length=steps)
    return jax.tree.map(lambda x: x[-1], outs), state

# Partition definition.
devices = jax.devices()
partition = spark.Partition(brain, {
    devices[0]: ['spiker'],
    devices[1]: ['integrator'],
    devices[2]: ['A_excitatory'],
    devices[3]: ['A_inhibitory'],
    devices[4]: ['B_excitatory'],
    devices[5]: ['B_inhibitory'],
})

# Split the graph with partition
graphs, states = partition.split(brain)
received = partition.initial(states)
for _ in range(100):
    outputs, states = partition.run(run, graphs, states, received, inputs, steps=1)
	# Exchange outputs between nodes
    received = partition.exchange(outputs)
# Merge the graph with partition
merged = partition.merge(states)

# Compare the output with a brain run on a single device.
graph, state = spark.split(brain)
for _ in range(100):
    _, state = run(graph, state, 1, **inputs)
same = all(np.array_equal(x, y) for x, y in zip(jax.tree.leaves(spark.split(merged)[1]), jax.tree.leaves(state)))

modules = [name for device in partition.local_devices for name in partition.parts[device]]
print(f'Process {process_id} ran {modules}. Same brain as on a single device: {same}')

Overwriting cluster.py


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Now, let's start the program twice, as process 0 and process 1, just like every machine of a cluster would. On a real cluster, every machine runs the very same command, <b>python cluster.py</b> followed by its process id, with the address of the first machine instead of <b>localhost</b>.

</div>

In [19]:
import subprocess
import sys

processes = [
    subprocess.Popen([sys.executable, 'cluster.py', str(process_id)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for process_id in range(2)
]
for process in processes:
    out, errors = process.communicate()
    # Gloo, which connects the processes, also prints on the output: we keep the last line, the one of our program.
    print(out.splitlines()[-1] if process.returncode == 0 else errors)

os.remove('cluster.py')

Process 0 ran ['spiker', 'integrator', 'A_excitatory']. Same brain as on a single device: True
Process 1 ran ['A_inhibitory', 'B_excitatory', 'B_inhibitory']. Same brain as on a single device: True


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Each process ran only half of the brain, and yet both of them ended up with the very same brain we get on a single device.

Note that, for now, recording a partition is only supported within one process.

</div>

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

A few things worth keeping in mind:

* On every step, a device needs what the devices it reads from produced on the step before. Therefore, the slowest device sets the pace of the whole brain: try to give every device a similar amount of work, and to keep together the modules that exchange the most (this is what <b>Partition.balanced</b> aims for).
* Partitioning is not free: every step adds one call per device and a copy of the outputs that cross devices. It pays off with large modules, whose steps take longer than this overhead; a small brain may well run faster on a single device.
* Properties are read and written within a step. A module that reads a property of another one, such as a plasticity rule reading the kernel of its synapses, or that writes one through an effect, has to live on the same device as that module; <b>spark.Partition</b> raises an error otherwise.

</div>